In [ ]:
import re
from pathlib import Path

import pandas as pd
from transitions.extensions import GraphMachine

# Configuration
DATA_PATH = Path("changes.csv")
MIN_TRANSITION_PROBABILITY = 0.05

EXPECTED_COLUMNS = [
    "repository",
    "commit_hash",
    "committed_date",
    "file_path",
    "git_change_type",
    "prev_file",
    "current_file",
    "job",
    "step",
    "from",
    "to",
    "change",
]

GROUP_KEYS = ["repository", "file_path", "job_categorized"]

extended_keywords = {
    "build": [
        "build", "compile", "assemble", "install", "package", "phar",
        "prepare", "run", "emscripten", "gauzy-webapp", "gauzy-api",
        "docker-push", "rpm", "deb", "get_runner", "develop",
        "deploy-website", "deploy-flat-web-dev", "Gradle", "gradle", "setup"
    ],
    "test": [
        "test", "unit", "e2e", "functional", "regression", "cypress-run",
        "verify", "check", "phpunit", "R-CMD-check", "validation",
        "performance", "check-package", "typecheck", "demo-scripts",
        "validate", "validate_version_bumps"
    ],
    "release": [
        "release", "publish", "publish-npm", "publish-java-sdk", "deploy", "push",
        "publish_archives", "deploy_to_npm", "deploy-flat-web-dev",
        "deploy_to_firebase", "deploy-snapshot", "Publish", "push_to_registry"
    ],
    "analyze": [
        "analyze", "static-analysis", "codacy-coverage-reporter",
        "gradle-dependency-detection", "license-checks", "checks", "check_runners",
        "analysis", "clang-tidy", "clang-debug", "cppcheck", "phpcs",
        "php-cs-fixer", "security-scan", "ts-check", "node-checks",
        "style-check", "phpstan", "rubocop", "psalm", "misspell",
        "finalize-percy", "check-code", "shellcheck", "coverage",
        "mirror", "report"
    ],
    "integration": [
        "integration", "e2e-deploy-macos", "e2e-deploy-windows",
        "run_models_gpu", "run_torch_cuda_extensions_gpu",
        "run_quantization_torch_gpu", "run_pipelines_torch_gpu",
        "run_pipelines_tf_gpu", "run_examples_gpu",
        "run_trainer_and_fsdp_gpu", "CI", "ci"
    ],
    "sync": [
        "sync", "fetch-dependencies", "update-gradle-wrapper",
        "update_phpstorm_meta", "update_copyright",
        "post-update", "clean", "mutate", "update", "rebase"
    ],
    "lint": ["lint"],
    "linux": ["linux", "ubuntu"],
}

CACHE_PROPERTIES = {
    "key", "path", "id", "restore-keys", "cache", "cache-dependency-path",
    "cache-from", "cache-to", "bundler-cache", "with"
}

SETUP_GO_DEFAULT_CACHE_RE = re.compile(r"(^|\s)actions/setup-go@\s*v(?:4|5|6)(?:\D|$)", re.I)


# Helpers
def normalize_text(value) -> str:
    if value is None or (pd.isna(value) and not isinstance(value, str)):
        return ""
    return str(value).strip()


def categorize_job(job_name: str) -> str:
    lower = normalize_text(job_name).lower()
    for category, kws in extended_keywords.items():
        for kw in kws:
            if kw.lower() in lower:
                return category
    return "other"


def is_cache_action(action_text: str) -> bool:
    if not isinstance(action_text, str):
        return False

    s = action_text.lower().strip()
    if not s:
        return False

    if "actions/cache@" in s or "uses: actions/cache" in s:
        return True

    # Setup Go:
    # - v4/v5/v6: caching is enabled by default
    # - any other version: require explicit 'cache:' parameter
    if "actions/setup-go@" in s:
        if SETUP_GO_DEFAULT_CACHE_RE.search(s):
            return True
        if "cache:" in s:
            return True
        return False

    # Node.js
    if "actions/setup-node@" in s:
        if "package-manager-cache: true" in s:
            return True
        if "cache:" in s and "package-manager-cache: false" not in s:
            return True
        return False

    # Python / Java
    if "actions/setup-python@" in s and "cache:" in s:
        return True
    if "actions/setup-java@" in s and "cache:" in s:
        return True

    # Ruby
    if "actions/setup-ruby@" in s:
        if "cache:" in s or "bundler-cache: true" in s:
            return True

    if "ruby/setup-ruby@" in s:
        return True

    return False


def cache_type(action_text: str) -> str | None:
    if not isinstance(action_text, str):
        return None

    s = action_text.lower().strip()

    if "actions/cache@" in s or "uses: actions/cache" in s:
        return "File/Folder Cache (actions/cache)"
    if "actions/setup-node@" in s:
        return "Node.js Package Manager Cache"
    if "actions/setup-python@" in s:
        return "Python Package Manager Cache"
    if "actions/setup-java@" in s:
        return "Java Package Manager Cache"
    if "actions/setup-go@" in s:
        return "Go Module Cache"
    if "actions/setup-ruby@" in s or "ruby/setup-ruby@" in s:
        return "Ruby Bundler Cache"

    return "Other/Unknown cache signal"


def classify_cache_change(row) -> str:
    if row["cache_transition"] in ["added", "removed"]:
        return row["cache_transition"]

    change = normalize_text(row["change"]).lower()

    if "key" in change:
        return "key changed"
    if "path" in change:
        return "path changed"
    if "uses" in change:
        return "uses changed"
    if "id" in change:
        return "id changed"
    if "restore-keys" in change:
        return "restore-keys changed"
    if "cache-dependency-path" in change:
        return "cache-dependency-path changed"
    if "cache-from" in change:
        return "cache-from changed"
    if "cache-to" in change:
        return "cache-to changed"
    if "bundler-cache" in change:
        return "bundler-cache changed"
    if "cache" in change:
        return "cache parameter changed"
    if "with" in change:
        return "with changed"

    return "other cache property changed"


def extract_path_key(value):
    if value is None or (pd.isna(value) and not isinstance(value, str)):
        return None, None

    value = str(value)

    m_path = re.search(r"""["']path["']\s*:\s*(["'])(.*?)\1""", value, flags=re.DOTALL)
    m_key  = re.search(r"""["']key["']\s*:\s*(["'])(.*?)\1""", value, flags=re.DOTALL)

    return (
        m_path.group(2) if m_path else None,
        m_key.group(2) if m_key else None
    )


def relabel_with_changes(row):
    ch = normalize_text(row.get("change"))
    up = normalize_text(row.get("updated_change"))

    is_with = bool(re.search(r"\bwith[_\s]+changed\b", ch, flags=re.I))
    needs = (up == "") or bool(re.search(r"\bwith[_\s]+changed\b", up, flags=re.I))

    if not (is_with and needs):
        return row

    from_path, from_key = extract_path_key(row.get("from"))
    to_path, to_key = extract_path_key(row.get("to"))

    path_changed = (from_path != to_path) and (from_path is not None or to_path is not None)
    key_changed = (from_key != to_key) and (from_key is not None or to_key is not None)

    if path_changed and key_changed:
        row["updated_change"] = "key_and_path_changed"
    elif path_changed:
        row["updated_change"] = "path_changed"
    elif key_changed:
        row["updated_change"] = "key_changed"
    else:
        row["updated_change"] = "with_changed"

    return row


# Load data
print("Loading input data...")
diffs_df = pd.read_csv(DATA_PATH, low_memory=False)

missing = [c for c in EXPECTED_COLUMNS if c not in diffs_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

diffs_df = diffs_df[EXPECTED_COLUMNS].copy()
diffs_df["committed_date"] = pd.to_datetime(diffs_df["committed_date"], errors="coerce")

for col in ["repository", "commit_hash", "file_path", "job", "step", "from", "to", "change"]:
    diffs_df[col] = diffs_df[col].apply(normalize_text)

diffs_df = diffs_df[
    (diffs_df["repository"] != "") &
    (diffs_df["file_path"] != "") &
    diffs_df["committed_date"].notna()
].copy()

diffs_df["job_categorized"] = diffs_df["job"].apply(categorize_job)


# Mark cache actions and cache-related rows
print("Detecting cache-related changes...")

diffs_df["is_cache_action"] = diffs_df.apply(
    lambda row: is_cache_action(row.get("from", "")) or is_cache_action(row.get("to", "")),
    axis=1
)

cache_steps = set()
for _, row in diffs_df[diffs_df["is_cache_action"]].iterrows():
    if row["step"]:
        cache_steps.add((row["repository"], row["file_path"], row["job_categorized"], row["step"]))

diffs_df["cache_related"] = False

for idx, row in diffs_df.iterrows():
    if row["is_cache_action"]:
        diffs_df.at[idx, "cache_related"] = True
        continue

    step_key = (row["repository"], row["file_path"], row["job_categorized"], row["step"])
    if step_key in cache_steps:
        change_text = normalize_text(row["change"]).lower()
        if any(prop in change_text for prop in CACHE_PROPERTIES):
            diffs_df.at[idx, "cache_related"] = True

diffs_df["from_has_cache"] = diffs_df["cache_related"] & (diffs_df["from"] != "")
diffs_df["to_has_cache"] = diffs_df["cache_related"] & (diffs_df["to"] != "")


def infer_transition(row):
    if row["from_has_cache"] and not row["to_has_cache"]:
        return "removed"
    if not row["from_has_cache"] and row["to_has_cache"]:
        return "added"
    if row["from_has_cache"] and row["to_has_cache"]:
        return "modified"
    return "no change"


diffs_df["cache_transition"] = diffs_df.apply(infer_transition, axis=1)


def determine_cache_type(row):
    if row["to_has_cache"] and isinstance(row["to"], str):
        return cache_type(row["to"])
    if row["from_has_cache"] and isinstance(row["from"], str):
        return cache_type(row["from"])
    return None


diffs_df["cache_type_signal"] = diffs_df.apply(determine_cache_type, axis=1)


# Keep only cache-related changes after first add
def collect_changes_for_group(group):
    group = group.sort_values(["committed_date", "commit_hash"])
    add_idx = group.index[group["cache_transition"] == "added"]

    if len(add_idx) == 0:
        return pd.DataFrame(columns=group.columns)

    first_add_time = group.loc[add_idx[0], "committed_date"]
    post = group[(group["committed_date"] >= first_add_time) & group["cache_related"]].copy()
    return post


changes_list = []
for (repo, fpath), group in diffs_df.groupby(["repository", "file_path"]):
    changes = collect_changes_for_group(group)
    if not changes.empty:
        changes_list.append(changes)

changes_df = pd.concat(changes_list, ignore_index=True) if changes_list else pd.DataFrame()

if changes_df.empty:
    raise ValueError("No cache-related changes found after first cache addition.")

changes_df["cache_change_type"] = changes_df.apply(classify_cache_change, axis=1)


# Build RQ2 state labels
changes_df["committed_date"] = pd.to_datetime(changes_df["committed_date"], errors="coerce")
sort_keys = ["repository", "file_path", "job_categorized", "step", "committed_date", "commit_hash"]
changes_df = changes_df.sort_values(sort_keys).copy()

changes_df["updated_change"] = changes_df["cache_change_type"]

added_mask = (
    changes_df["change"].astype(str).str.contains(r"\badded\b", case=False, na=False) &
    changes_df["to"].astype(str).str.contains(r"cache", case=False, na=False)
)

grp = ["repository", "file_path", "job_categorized"]
first_commits = (
    changes_df.loc[added_mask]
    .sort_values(grp + ["committed_date", "commit_hash"])
    .drop_duplicates(grp, keep="first")
    .set_index(grp)["commit_hash"]
)

row_grp_index = pd.MultiIndex.from_frame(changes_df[grp])
earliest_commit_for_row = first_commits.reindex(row_grp_index).to_numpy()

same_commit_first_enable = added_mask & (changes_df["commit_hash"].to_numpy() == earliest_commit_for_row)
changes_df.loc[same_commit_first_enable, "updated_change"] = "enable_cache"
changes_df.loc[added_mask & ~same_commit_first_enable, "updated_change"] = "cache added"

changes_df = changes_df.apply(relabel_with_changes, axis=1)

changes_df["change"] = changes_df["change"].str.replace(
    r"\bwith[_\s]+changed\b", "with_changed", case=False, regex=True
)

changes_df["updated_change"] = changes_df["updated_change"].replace({
    "cache added": "adding_cache",
    "cache changed": "cache_version_update",
    "cache removed": "removing_cache",
    "parameter changed": "updating_parameter",
    "parameter added": "adding_parameter",
    "parameter removed": "removing_parameter",
    "cache_added": "adding_cache",
    "key changed": "updating_parameter",
    "path changed": "updating_parameter",
    "id changed": "updating_parameter",
    "restore-keys changed": "updating_parameter",
    "cache parameter changed": "updating_parameter",
    "bundler-cache changed": "updating_parameter",
    "cache-from changed": "updating_parameter",
    "cache-to changed": "updating_parameter",
    "cache-dependency-path changed": "updating_parameter",
    "with_changed": "updating_parameter",
    "key_changed": "updating_parameter",
    "path_changed": "updating_parameter",
    "key_and_path_changed": "updating_parameter",
    "other cache property changed": "updating_parameter",
    "added": "adding_parameter",
    "removed": "removing_parameter",
    "uses changed": "cache_version_update",
})


# Build transitions
changes_df = changes_df.sort_values(GROUP_KEYS + ["committed_date", "commit_hash"]).copy()
changes_df["from_state"] = changes_df["updated_change"]
changes_df["to_state"] = changes_df.groupby(GROUP_KEYS)["from_state"].shift(-1)
changes_df["next_time"] = changes_df.groupby(GROUP_KEYS)["committed_date"].shift(-1)

transitions = changes_df.dropna(subset=["to_state"]).copy()
transitions["time_to_next_days"] = (
    (transitions["next_time"] - transitions["committed_date"]).dt.total_seconds() / 86400.0
)

transitions_use = transitions.copy()

job_summary = (
    transitions_use
    .groupby(["job_categorized", "from_state", "to_state"], dropna=False)
    .agg(
        count=("to_state", "size"),
        avg_time_days=("time_to_next_days", "mean"),
        median_time_days=("time_to_next_days", "median")
    )
    .reset_index()
)

job_summary["probability"] = (
    job_summary["count"] /
    job_summary.groupby(["job_categorized", "from_state"])["count"].transform("sum")
)

job_summary = job_summary[job_summary["probability"] > MIN_TRANSITION_PROBABILITY].copy()

tables_by_job = {
    job: sub.sort_values(
        ["count", "probability", "from_state", "to_state"],
        ascending=[False, False, True, True]
    ).reset_index(drop=True)
    for job, sub in job_summary.groupby("job_categorized", sort=False)
}


# Build graphs
job_graphs = {}

for job, sub in tables_by_job.items():
    states_j = sorted(set(sub["from_state"]).union(set(sub["to_state"])))
    edges = [
        {
            "trigger": f"p={r['probability']:.2f} | median={r['median_time_days']:.2f}d",
            "source": r["from_state"],
            "dest": r["to_state"]
        }
        for _, r in sub.iterrows()
    ]

    model = type("Model", (), {})()
    machine = GraphMachine(
        model=model,
        states=states_j,
        transitions=edges,
        initial=states_j[0] if states_j else None,
        auto_transitions=False
    )

    job_graphs[job] = machine.get_graph()



In [ ]:
import pandas as pd


def summarize_from_states_for_fixed_to(df_for_state: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize transitions into one fixed to_state.

    Parameters
    ----------
    df_for_state : pd.DataFrame
        Detailed transition rows for a single fixed to_state.
        Expected columns:
        - from_state
        - to_state
        - time_to_next_days

    Returns
    -------
    pd.DataFrame
        Summary by from_state with:
        - to_state
        - from_state
        - count
        - avg_time_days
        - median_time_days
        - std_time_days
        - min_time_days
        - max_time_days
        - percent
    """
    expected_cols = {"from_state", "to_state", "time_to_next_days"}
    missing = expected_cols - set(df_for_state.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    if df_for_state.empty:
        return pd.DataFrame(columns=[
            "to_state",
            "from_state",
            "count",
            "avg_time_days",
            "median_time_days",
            "std_time_days",
            "min_time_days",
            "max_time_days",
            "percent",
        ])

    unique_to_states = df_for_state["to_state"].dropna().unique()
    if len(unique_to_states) == 0:
        fixed_to_state = None
    elif len(unique_to_states) == 1:
        fixed_to_state = unique_to_states[0]
    else:
        raise ValueError(
            "summarize_from_states_for_fixed_to expects exactly one to_state in df_for_state."
        )

    grouped = (
        df_for_state
        .groupby("from_state", dropna=False)["time_to_next_days"]
        .agg(
            count="size",
            avg_time_days="mean",
            median_time_days="median",
            std_time_days="std",
            min_time_days="min",
            max_time_days="max",
        )
        .reset_index()
    )

    total = grouped["count"].sum()
    grouped["percent"] = (grouped["count"] / total * 100).round(2)

    time_cols = [
        "avg_time_days",
        "median_time_days",
        "std_time_days",
        "min_time_days",
        "max_time_days",
    ]
    grouped[time_cols] = grouped[time_cols].round(2)

    grouped.insert(0, "to_state", fixed_to_state)

    grouped = grouped.sort_values(
        ["count", "percent", "from_state"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    return grouped


def build_state_views(to_states, transitions_by_to_state_fn):
    """
    Build both detailed and summary views for multiple target to_states.

    Parameters
    ----------
    to_states : list[str]
        List of target states to analyze.
    transitions_by_to_state_fn : callable
        Function that accepts one to_state and returns the detailed transition rows.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        from_to_summary_all, df_all_jobs_for_states
    """
    detail_frames = []
    summary_frames = []

    for to_state in to_states:
        df_state = transitions_by_to_state_fn(to_state)

        if df_state is None or df_state.empty:
            print(f"[warn] No rows found for to_state='{to_state}'. Skipping.")
            continue

        df_state = df_state.copy()

        if "to_state" not in df_state.columns:
            df_state["to_state"] = to_state

        summary_df = summarize_from_states_for_fixed_to(df_state)

        detail_frames.append(df_state)
        summary_frames.append(summary_df)

    if detail_frames:
        df_all_jobs_for_states = pd.concat(detail_frames, ignore_index=True)
    else:
        df_all_jobs_for_states = pd.DataFrame(columns=[
            "repository",
            "file_path",
            "job_categorized",
            "step",
            "committed_date",
            "from_state",
            "to_state",
            "time_to_next_days",
            "from_commit_hash",
            "to_commit_hash",
        ])

    if summary_frames:
        from_to_summary_all = pd.concat(summary_frames, ignore_index=True)
    else:
        from_to_summary_all = pd.DataFrame(columns=[
            "to_state",
            "from_state",
            "count",
            "avg_time_days",
            "median_time_days",
            "std_time_days",
            "min_time_days",
            "max_time_days",
            "percent",
        ])

    return from_to_summary_all, df_all_jobs_for_states


TO_STATES = [
    "cache_version_update",
    "updating_parameter",
    "adding_cache",
    "adding_parameter",
    "removing_cache",
    "removing_parameter",
]

from_to_summary_all, df_all_jobs_for_states = build_state_views(
    to_states=TO_STATES
)



In [ ]:
from_to_summary_all

,from_state,to_state,count,avg_time_days,median_time_days,std_time_days,min_time_days,max_time_days,percent
0,cache_version_update,cache_version_update,378,70.60,0.00,149.79,0.00,713.59,51.85
1,enable_cache,cache_version_update,168,543.33,384.13,447.09,0.01,1615.93,23.05
2,updating_parameter,cache_version_update,104,172.52,104.80,241.06,0.00,1366.00,14.27
3,adding_cache,cache_version_update,44,298.89,231.83,310.83,0.00,1294.85,6.04
4,removing_cache,cache_version_update,18,210.66,95.11,334.78,0.00,1374.41,2.47
5,adding_parameter,cache_version_update,12,179.04,0.00,392.71,0.00,1366.00,1.65
6,removing_parameter,cache_version_update,5,0.00,0.00,0.00,0.00,0.00,0.69
7,updating_parameter,updating_parameter,497,75.36,4.29,148.32,0.00,986.95,58.20
8,enable_cache,updating_parameter,166,207.31,115.90,260.34,0.00,981.44,19.44
9,cache_version_update,updating_parameter,92,95.24,0.23,136.98,0.00,592.69,10.77
